## How to write SQL Query using Frame Clause, CUME_DIST 

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS sql.advance;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS sql.advance.product (
  product_category STRING,
  brand STRING,
  product_name STRING,
  price DECIMAL(10,2)
);

SELECT * FROM sql.advance.product;

In [0]:
%sql
INSERT INTO sql.advance.product (product_category, brand, product_name, price) VALUES
  ('Electronics', 'Apple', 'iPhone 15', 999.00),
  ('Electronics', 'Apple', 'MacBook Air', 1199.00),
  ('Electronics', 'Samsung', 'Galaxy S24', 849.00),
  ('Electronics', 'Sony', 'WH-1000XM5 Headphones', 399.99),
  ('Furniture', 'IKEA', 'MARKUS Office Chair', 199.00),
  ('Furniture', 'Herman Miller', 'Aeron Chair', 1395.00),
  ('Furniture', 'West Elm', 'Mid-Century Desk', 749.00),
  ('Groceries', 'Organic Valley', 'Whole Milk 1gal', 4.99),
  ('Groceries', 'General Mills', 'Cheerios 18oz', 5.49),
  ('Groceries', 'Kraft', 'Mac & Cheese', 1.29);

SELECT * FROM sql.advance.product;

#### first_value()

The SQL FIRST_VALUE window function returns the first value in an ordered partition, using syntax like FIRST_VALUE(column) OVER (PARTITION BY ... ORDER BY ...).

In [0]:
%sql
--Write query to display the most expensive product under each category(corresponding to each record).
SELECT p.*,
first_value(product_name) over(PARTITION BY product_category ORDER BY price DESC) AS most_expensive_product FROM sql.advance.product p;

#### last_value()

LAST_VALUE returns the value from the last row of the current window frame, and to get the true last value of a partition you must explicitly set the frame to ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING.

* What LAST_VALUE Does?

LAST_VALUE(expr) is a window (analytic) function that returns the value of expr from the last row within the window frame, not necessarily the last row of the partition.

* frame clause:

The ROWS or RANGE clause that bounds the frame. Almost always required in practice: use ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING to reach the true last value.

Note: Frame clause: essential for correct results.

#### Definitions

**ROWS**: This clause defines a window frame based on the physical order of rows. It includes a specified number of rows that precede or follow the current row, regardless of their values. For example, using ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING will include the current row and one row before and after it.

**RANGE**: This clause defines a window frame based on logical values. It includes all rows that have the same value as the current row, as well as rows that fall within a specified range of values. For example, RANGE BETWEEN 10 PRECEDING AND 10 FOLLOWING will include all rows with values within 10 units of the current row's value.

In [0]:
%sql
--Write query to display the least expensive product under each category(corresponding to each record).
SELECT p.*,
first_value(product_name) over(PARTITION BY product_category ORDER BY price DESC) AS most_expensive_product,
last_value(product_name) over(PARTITION BY product_category ORDER BY price desc range between unbounded preceding and unbounded following) AS least_expensive_product FROM sql.advance.product p;

In [0]:
%sql
--Alternate way to write SQL query using Window functions
SELECT p.*,
first_value(product_name) over w AS most_expensive_product,
last_value(product_name)  over w AS least_expensive_product 
FROM sql.advance.product p
window w as (PARTITION BY product_category ORDER BY price desc range between unbounded preceding and unbounded following);

#### Nth Value

NTH_VALUE() in SQL returns the value from the nth row of a window frame, defined by OVER(), using an ordered set of rows. It requires an ORDER BY clause and often a full-frame specification like ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING for consistent results.

* What NTH_VALUE() Does?

NTH_VALUE(expr, n) returns the value of expr from the nth row in the window frame:

- n is a positive integer (1 = first row).

- Returns NULL if the frame has fewer than n rows.

In [0]:
%sql
--Find the second most expensive product in each category(corresponding to each record)
SELECT p.*,
first_value(product_name) over w AS most_expensive_product,
last_value(product_name)  over w AS least_expensive_product,
nth_value(product_name,2) over w AS second_most_expensive_product
FROM sql.advance.product p
window w as (PARTITION BY product_category ORDER BY price desc range between unbounded preceding and unbounded following);

#### NTILE

NTILE in SQL is a window function that divides ordered rows into n roughly equal groups (tiles) and assigns each row a bucket number from 1 to n. It requires an OVER(ORDER BY ...) clause and is commonly used to create quartiles, deciles, or percentiles.

* What NTILE Does?

NTILE distributes rows into a specified number of groups after ordering them:

NTILE(n) → splits rows into n buckets.

Buckets are as equal as possible; if rows don’t divide evenly, earlier buckets get one extra row. 

Returns an integer bucket number between 1 and n.

In [0]:
%sql
--Write a query to segregate all the expensive phones,mid range phones and the cheaper phones.

SELECT *,
CASE WHEN x.buckets = 1 THEN 'Expensive Phones'
           WHEN x.buckets = 2 THEN 'Mid Range Phones'
           WHEN x.buckets = 3 THEN 'Cheaper Phones' END AS phone_category
FROM (
	SELECT *,
	ntile(3) OVER (ORDER BY price DESC) AS buckets
	FROM sql.advance.product
	WHERE product_category='Electronics') x;

#### CUM_DIST(Cumulative Distribution)

The CUME_DIST() function in SQL is a window function that calculates the cumulative distribution of a value within a group of values. It returns a value between 0 and 1, where 0 represents the first row and 1 represents the last row in the result set. The function is used to determine the relative position of a specified value in a group of values, such as the percent of employees with a salary less than or equal to the current employee's salary within a department.



In [0]:
%sql
/*Value --> 1<= CUME_DIST >0 */
/* Formula = Current  Row no (or Row No with value same as current row) / Total no of rows */

-- Query to fetch all products which are constituting the first 30% of the data in products table based on price.

SELECT product_name, concat(cume_dist_percentage,'%')  AS cume_dist_percentage
FROM 
	(SELECT *,
	 cume_dist() over (ORDER BY price DESC) AS cume_distribution,
	round(cume_dist() OVER (ORDER BY price DESC)::NUMERIC *100,2) AS cume_dist_percentage FROM sql.advance.product) x
WHERE x.cume_dist_percentage <=30; 

#### PERCENT_RANK

PERCENT_RANK is a SQL window function that returns a row’s relative rank within a result set or partition, calculated as (rank - 1) / (total_rows - 1) and producing a value between 0 and 1.

PERCENT_RANK is used to measure how a row compares to others in the same ordered group. It expresses rank as a percentage rather than an integer position.

* Range: 0 to 1 (inclusive)
* First row: Always returns 0
* Purpose: Shows relative standing, useful for percentile-like analysis
* Similar to: CUME_DIST, but uses rank rather than cumulative distribution 

In [0]:
%sql
-- PERCENT_RANK(relative rank of the current row / Percentage Ranking)
/* Value --> 1 <= PERCENT_RANK >0 */
/* Formula = Current Row No -1 /Total no of rows -1 */

SELECT product_name, concat(percentage_rank,'%')  AS  percentage_rank
FROM 
	(SELECT *,
	 percent_rank() over (ORDER BY price DESC) AS percentage_rank,
	round(percent_rank() OVER (ORDER BY price DESC)::NUMERIC *100,2) AS per_rank FROM sql.advance.product) x
WHERE x.percentage_rank <=30; 
